#### Extraction des caractéristiques avec model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16_version_2.pt

In [ ]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models
from tqdm import tqdm
import warnings
import gc
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

# Configuration des chemins
model_path = r"D:\wealth_predict_sentinel\models\model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt" 
test_image_dir = r"D:\wealth_predict_sentinel\Data\downloaded\Image_satellite_base_EHCVM_2018_Zoom_14_Sentinel_2_pour_an_2023" # menage EHCVM 2018 leurs images en 2023 
csv_path = r"D:\wealth_predict_sentinel\Data\processed_csv\Data_EHCVM_2018_with_images_names.csv"
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16_base_EHCVM_2018_pour_an_2023_version_2.csv"

# Configuration du device et des paramètres
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 32
print(f"Utilisation du device: {device}")

# Architecture du modèle identique à l'entraînement
class VGGFullyConv(nn.Module):
    def __init__(self, num_classes=4):
        super(VGGFullyConv, self).__init__()
        self.features = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, num_classes, kernel_size=1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier[0](x)  # Première Conv2d (512 -> 4096)
        x = self.classifier[1](x)  # Premier ReLU
        x = self.classifier[2](x)  # Deuxième Conv2d (4096 -> 4096)
        x = self.classifier[3](x)  # Deuxième ReLU
        return x.mean(dim=(2, 3))  # Moyennage spatial pour obtenir un vecteur de dimension 4096

# Transformation des images identique à l'entraînement
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def load_image(image_path):
    """Charge et transforme une image."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            return transform(img)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

def batch_extract_features(model, image_tensors):
    """Extrait les caractéristiques pour un lot d'images."""
    with torch.no_grad():
        features = model(image_tensors)
    return features.cpu().numpy()

def process_images_in_batches(image_paths, model, batch_size):
    """Traite les images par lots pour optimiser la mémoire."""
    features_list = []
    current_batch = []
    
    for img_path in tqdm(image_paths, desc="Traitement des images"):
        if os.path.exists(img_path):
            img_tensor = load_image(img_path)
            if img_tensor is not None:
                current_batch.append(img_tensor)
                
                if len(current_batch) >= batch_size:
                    # Traitement du lot
                    batch_tensor = torch.stack(current_batch).to(device)
                    batch_features = batch_extract_features(model, batch_tensor)
                    features_list.extend(batch_features)
                    
                    # Nettoyage de la mémoire
                    current_batch = []
                    del batch_tensor
                    torch.cuda.empty_cache()
                    gc.collect()
            else:
                features_list.append(np.zeros(4096))
        else:
            print(f"Image introuvable: {img_path}")
            features_list.append(np.zeros(4096))
    
    # Traitement du dernier lot
    if current_batch:
        batch_tensor = torch.stack(current_batch).to(device)
        batch_features = batch_extract_features(model, batch_tensor)
        features_list.extend(batch_features)
        del batch_tensor
        torch.cuda.empty_cache()
    
    return features_list

def main():
    try:
        # 1. Chargement et configuration du modèle
        print("Chargement du modèle...")
        model = VGGFullyConv(num_classes=4).to(device)
        model.load_state_dict(torch.load(model_path), strict=False)
        model.eval()

        # 2. Vérification de la sortie du modèle
        print("Vérification des dimensions des caractéristiques...")
        dummy_image = torch.randn(1, 3, 224, 224).to(device)
        with torch.no_grad():
            dummy_features = model(dummy_image)
        print(f"Dimensions des caractéristiques: {dummy_features.shape}")
        del dummy_image, dummy_features
        torch.cuda.empty_cache()

        # 3. Chargement des données
        print("Chargement des données...")
        df = pd.read_csv(csv_path)
        total_images = len(df)
        print(f"Nombre total d'images à traiter: {total_images}")

        # 4. Préparation des chemins d'images
        image_paths = [os.path.join(test_image_dir, name) for name in df['nom de l\'image']]

        # 5. Extraction des caractéristiques
        print("Début de l'extraction des caractéristiques...")
        features_list = process_images_in_batches(image_paths, model, batch_size)

        # 6. Création du DataFrame final
        print("Création du DataFrame avec les caractéristiques...")
        features_df = pd.DataFrame(
            features_list,
            columns=[f"feature_{i}" for i in range(4096)]
        )

        # 7. Combinaison avec le DataFrame original
        df_with_features = pd.concat(
            [df.reset_index(drop=True), features_df.reset_index(drop=True)],
            axis=1
        )

        # 8. Sauvegarde des résultats
        print("Sauvegarde des résultats...")
        df_with_features.to_csv(output_path, index=False)
        print(f"Extraction terminée. Résultats sauvegardés dans: {output_path}")
        print(f"Nombre total de caractéristiques extraites: {len(features_list)}")

    except Exception as e:
        print(f"Une erreur est survenue: {str(e)}")
        import traceback
        traceback.print_exc()
    finally:
        # Nettoyage final de la mémoire
        torch.cuda.empty_cache()
        gc.collect()

if __name__ == "__main__":
    main()

In [ ]:
pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConv_sans_augm_couche_nongelee_batch_16_base_EHCVM_2018_pour_an_2023_version_2.csv")